In [3]:
import yaml
import json
import pickle
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import httpx
import os
import ast
import krippendorff


import warnings
warnings.filterwarnings("ignore")

from prolific import ProlificAPIError, ProlificClient

pd.set_option('display.max_columns', None)

# config = yaml.safe_load(open("config.yaml", "r"))

client = ProlificClient(
    api_token=os.environ["prolific_key"])

In [4]:
ID = "698efef38f6dc0baf73e7f6f"

In [5]:
client.get_study(ID)

{'id': '698efef38f6dc0baf73e7f6f',
 'name': 'Evaluating the Persuasiveness of Debaters',
 'description': '<p>In this study, you will evaluate <strong>persuasiveness in debate transcripts</strong>. You will read transcripts from <strong>two separate debates</strong> on the same topic and judge which speaker is more persuasive.</p><p>Each debate features two participants, Speaker A and Speaker B. Although the debates involve different individuals, <strong>Speaker A represents the same ideological or political position in both debates</strong>.</p><p>Your task is to compare the persuasiveness of <strong>Speaker A across the two debates</strong>, focusing only on how effectively they argue their position.</p><p>There are <strong>no right or wrong answers</strong>! We are interested in your honest, subjective judgment based on the debate content.</p><p></p>',
 'total_available_places': 3234,
 'reward': 120,
 'average_reward_per_hour': 1155.0,
 'external_study_url': 'https://tars.dcp.prod.aw

In [6]:
responses = client.list_submissions(study=ID, page_size=1000)
print(len(responses["results"]))
print(json.dumps(responses['results'][0], indent=2))

1000
{
  "id": "698f546d256d83fbae917ec5",
  "participant_id": "673b7a139cd7db8678fafdcd",
  "started_at": "2026-02-13T16:42:42.802000Z",
  "completed_at": "2026-02-13T16:43:55.622000Z",
  "is_complete": true,
  "time_taken": 72,
  "reward": 12000,
  "status": "AWAITING REVIEW",
  "strata": {},
  "study_code": "NOCODE",
  "bonus_payments": [],
  "ip": "194.233.158.124",
  "return_requested": null,
  "has_siblings": false,
  "time_taken_under_auto_approval_threshold": true,
  "dynamic_payment_percentage": null
}


In [8]:
rejected = client.list_submissions(study=ID, page_size=1000, rejected=True)
print(len(rejected["results"]))

5


In [9]:
rejected_ids = [sub['participant_id'] for sub in rejected['results']]
rejected_ids

['5b70252c99982e000145ccf7',
 '5cd80c45dcabe80001040e88',
 '5d78d57c8f579d001518e74f',
 '5fcfa3168335430d143b4431',
 '67c3321da0a96b0538b4abbe']

In [10]:


# --- 1. Reload annotator columns (before they were dropped) ---
# Re-read the original data
df_raw = pd.read_csv("data/019c5684-e8e6-737c-9442-07692f856379.csv")

# Remove attention checks
df_raw = df_raw[~df_raw["Debate Topic"].str.contains("ATTENTION CHECK:", na=False)]

# Remove rejected annotator responses
for idx, row in df_raw.iterrows():
    for col in df_raw.columns:
        if col.startswith("Annotator") and col.endswith("_ID"):
            if row[col] in rejected_ids:
                base = col.replace("_ID", "")
                df_raw.at[idx, col] = np.nan
                df_raw.at[idx, base + "_Timestamp"] = np.nan
                df_raw.at[idx, base + "_Response"] = np.nan

# --- 2. Persuasiveness (ordinal or nominal) ---
persuasiveness_df = df_raw[
    df_raw["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"
].copy()

# Map responses to ordinal codes
persuasiveness_map = {
    "Speaker A from Debate 1": 1,
    "Both Speaker As were equally persuasive": 2,
    "Speaker A from Debate 2": 3,
}

response_cols = [col for col in persuasiveness_df.columns if col.endswith("_Response")]

# Build reliability matrix: rows = annotators, columns = items
reliability_persuasiveness = []
for col in response_cols:
    coded = persuasiveness_df[col].map(persuasiveness_map).values.astype(float)
    reliability_persuasiveness.append(coded)

reliability_persuasiveness = np.array(reliability_persuasiveness)

# Compute alpha (use 'nominal' since the 3 categories have no natural ordering,
# or 'ordinal' if you consider Debate1 < Tie < Debate2 as ordered)
alpha_persuasiveness_nominal = krippendorff.alpha(
    reliability_data=reliability_persuasiveness,
    level_of_measurement="nominal"
)
alpha_persuasiveness_ordinal = krippendorff.alpha(
    reliability_data=reliability_persuasiveness,
    level_of_measurement="ordinal"
)

print(f"Krippendorff's alpha (persuasiveness, nominal): {alpha_persuasiveness_nominal:.4f}")
print(f"Krippendorff's alpha (persuasiveness, ordinal): {alpha_persuasiveness_ordinal:.4f}")

# --- 3. Confidence (ordinal) ---
confidence_df = df_raw[
    df_raw["Question"] == "How confident are you in your choice?"
].copy()

likert_map = {
    "Very unsure": 1,
    "Somewhat unsure": 2,
    "Neutral": 3,
    "Somewhat confident": 4,
    "Very confident": 5,
}

reliability_confidence = []
for col in response_cols:
    coded = confidence_df[col].map(likert_map).values.astype(float)
    reliability_confidence.append(coded)

reliability_confidence = np.array(reliability_confidence)

alpha_confidence_ordinal = krippendorff.alpha(
    reliability_data=reliability_confidence,
    level_of_measurement="ordinal"
)

print(f"Krippendorff's alpha (confidence, ordinal):     {alpha_confidence_ordinal:.4f}")

Krippendorff's alpha (persuasiveness, nominal): 0.1882
Krippendorff's alpha (persuasiveness, ordinal): 0.2619
Krippendorff's alpha (confidence, ordinal):     0.0594
